In [1]:
## Load libraries
import pandas as pd
import os
import rasterio
import numpy as np

import os
os.getcwd()

'/Users/henryoliver/Documents/MEDS/capstone/ET_GW_analysis_workflow/cultivation_groundwater'

Set the side length of the bounding box that you want to perform analysis over
For 1km x 1km bounding box, set 'bbox_side_length = 1"
For 2km x 2km bounding box, set 'bbox_side_length = 2" etc.

In [20]:
# Set bounding box side length (km)
bbox_side_length = 10

Read in groundwater data. Esnure that 'clean_site_waterdata_05072026.csv' file is in data/gw_data

In [21]:
# Read in groundwater data
df = pd.read_csv("../data/gw_data/clean_site_waterdata_05072026.csv")
df['latitude'] = df['latitude'].round(5) # Round latitude and longitude, necessary for later join
df['longitude'] = df['longitude'].round(4)

Next, we'll determine the classes that we'll count as 'cultivated' and 'non-cultivated' These values come from the USDA CDL legend located at the link below

https://www.nass.usda.gov/Research_and_Science/Cropland/docs/US_CDL_Legend.png

In [22]:
# Set categories for land use

# Cultivated: codes 1-60 (minus 61 fallow), 66-77 (tree crops), 204-254 (specialty crops)
cultivated_codes = set(
    list(range(1, 61)) +
    list(range(66, 78)) +
    list(range(204, 255))
) - {61} # fallow

Next, we extract the latitude and longitude values from our groundwater data, and access all of the cultivation data available for each well. This data is available for all years 2008-2020

In [23]:
## Results for all years 2008-2020


from pyproj import Transformer
from rasterio.windows import from_bounds
import rasterio

all_results = []
for year in range(2008, 2021):
    print(f"Processing {year}...")
    df_year = df[df['year_value'] == year].copy()
    if len(df_year) == 0:
        continue
    tif_path = f"../data/cultivation/{year}_30m_cdls/{year}_30m_cdls.tif"
    with rasterio.open(tif_path) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        for _, row in df_year.iterrows():
            x, y = transformer.transform(row['longitude'], row['latitude'])
            bbox = (x - (500*bbox_side_length), y - (500*bbox_side_length), x + (500*bbox_side_length), y + (500*bbox_side_length))
            window = from_bounds(*bbox, transform=src.transform)
            block = src.read(1, window=window)

            total = block.size
            cultivated = np.sum(np.isin(block, list(cultivated_codes)))
         

            all_results.append({
                'latitude': round(row['latitude'], 5),
                'longitude': round(row['longitude'], 4),
                'year_value': year,
                'pct_cultivated': cultivated / total * 100,
            })

all_results_df = pd.DataFrame(all_results)

all_results_df = all_results_df.drop_duplicates(subset=['latitude', 'longitude', 'year_value'])


all_results_df.head()

Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...


,latitude,longitude,year_value,pct_cultivated
0,37.31502,-100.8505,2008,60.459559
1,37.34495,-101.6104,2008,48.077808
2,37.42949,-100.2434,2008,59.892325
3,37.56096,-98.0580,2008,50.335020
4,37.59874,-100.9497,2008,54.389525


Now we have percent of cultivation for each lat/lon point in our groundwater data. Now we need to join this back to groundwater data to get everything together

In [24]:
df_final = df.merge(all_results_df, on=['latitude', 'longitude', 'year_value'], how='left')
gwl_cult_nkm = df_final[['year_value', 'site_id', 'depth_to_gw_ft', 'depth_to_gw_m', 'latitude', 'longitude', 'data_source', 'region', 'pct_cultivated']]
gwl_cult_nkm.tail()

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,pct_cultivated
986,2015,southern_willcox,132.500004,40.38600,31.36597,-109.663,Jasechko,Southwest US,0.101002
987,2016,southern_willcox,134.700004,41.05656,31.36597,-109.663,Jasechko,Southwest US,0.669138
988,2018,southern_willcox,147.000005,44.80560,31.36597,-109.663,Jasechko,Southwest US,2.407813
989,2019,southern_willcox,153.000005,46.63440,31.36597,-109.663,Jasechko,Southwest US,2.098495
990,2020,southern_willcox,154.700005,47.15256,31.36597,-109.663,Jasechko,Southwest US,1.550199


Now we output the joined dataframe to a CSV. This will end up in the 'outputs' folder within the home repository

In [25]:
gwl_cult_nkm.to_csv(f"../outputs/gwl_cult_{bbox_side_length}km.csv", index=False)